# Phase 4: Open-Source Model Benchmarking\n\nBenchmark editing models (InstructPix2Pix, MagicBrush, UltraEdit) across English, Hindi, Bangla, and Nepali prompts using the trained error classifier.\n\n**Input:** Phase 3 model + Phase 2 data\n**Output:** `benchmark_results.json` + `leaderboard.csv`

In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate clean-fid open_clip_torch torch torchvision pandas matplotlib

In [ ]:
# Cell 2: Config & Imports
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# ── Path Detection ──
IS_KAGGLE = os.path.exists("/kaggle/working")
OUTPUT_DIR = "/kaggle/working" if IS_KAGGLE else "./output"
INPUT_DIR = "/kaggle/input" if IS_KAGGLE else "./output"

PHASE2_DIR = os.path.join(INPUT_DIR, "phase2-output") if IS_KAGGLE else OUTPUT_DIR
PHASE3_DIR = os.path.join(INPUT_DIR, "phase3-output") if IS_KAGGLE else OUTPUT_DIR
PHASE1_DIR = os.path.join(INPUT_DIR, "phase1-output") if IS_KAGGLE else OUTPUT_DIR

# ── Constants ──
ERROR_TAXONOMY = {
    0: "Wrong Object", 1: "Missing Object", 2: "Extra Object",
    3: "Wrong Attribute", 4: "Spatial Error", 5: "Style Mismatch",
    6: "Over-editing", 7: "Under-editing", 8: "Artifact/Quality",
    9: "Ambiguous Prompt", 10: "Failed Removal",
}
NUM_ERROR_CLASSES = 11

BENCHMARK_MODELS = {
    "instructpix2pix": "timbrooks/instruct-pix2pix",
    "magicbrush": "osunlp/InstructPix2Pix-MagicBrush",
    "ultraedit": "BleachNick/UltraEdit",
}
BENCHMARK_LANGUAGES = ["en", "hi", "bn", "ne"]
BENCHMARK_NUM_SAMPLES = 500  # subset for benchmarking
IMAGE_SIZE = 224
BLIP2_MODEL = "Salesforce/blip2-opt-2.7b"
XLM_ROBERTA_MODEL = "xlm-roberta-base"
RANDOM_SEED = 42

GEN_IMAGE_DIR = os.path.join(OUTPUT_DIR, "generated_images")
os.makedirs(GEN_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Cell 3: Load trained classifier from Phase 3
from transformers import Blip2Model, Blip2Processor, XLMRobertaModel, XLMRobertaTokenizer
from torchvision import transforms


class DualImageAttention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
    
    def forward(self, edited_feat, original_feat):
        q = edited_feat.unsqueeze(1)
        kv = original_feat.unsqueeze(1)
        attn_out, _ = self.attn(q, kv, kv)
        return self.norm(attn_out.squeeze(1) + edited_feat)


class DualImageTextClassifier(nn.Module):
    def __init__(self, num_classes=NUM_ERROR_CLASSES, vision_dim=1408, text_dim=768):
        super().__init__()
        self.vision_proj = nn.Linear(vision_dim, 512)
        self.text_proj = nn.Linear(text_dim, 512)
        self.delta_attention = DualImageAttention(512, num_heads=8)
        self.classifier = nn.Sequential(
            nn.Linear(512 * 4, 512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )
    
    def forward(self, orig_vision_feat, edit_vision_feat, text_feat):
        orig_proj = self.vision_proj(orig_vision_feat)
        edit_proj = self.vision_proj(edit_vision_feat)
        text_proj = self.text_proj(text_feat)
        delta = self.delta_attention(edit_proj, orig_proj)
        fused = torch.cat([orig_proj, edit_proj, delta, text_proj], dim=-1)
        return self.classifier(fused)


# Load BLIP-2 vision encoder
print("Loading BLIP-2 vision encoder...")
blip2_model_full = Blip2Model.from_pretrained(BLIP2_MODEL, torch_dtype=torch.float16)
blip2_vision = blip2_model_full.vision_model.to(device).half()
blip2_vision.eval()
del blip2_model_full
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Load XLM-RoBERTa
print("Loading XLM-RoBERTa...")
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained(XLM_ROBERTA_MODEL)
xlm_model = XLMRobertaModel.from_pretrained(XLM_ROBERTA_MODEL).to(device)
xlm_model.eval()

# Load classifier weights
print("Loading trained classifier...")
classifier = DualImageTextClassifier().to(device)
checkpoint_path = os.path.join(PHASE3_DIR, "best_model", "best_checkpoint.pt")
checkpoint = torch.load(checkpoint_path, map_location=device)
classifier.load_state_dict(checkpoint["classifier"])
xlm_model.load_state_dict(checkpoint["xlm_model"])
classifier.eval()
print(f"Classifier loaded from epoch {checkpoint['epoch']+1}")

# Image transform
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 4: Load test set from Phase 2/3
df = pd.read_parquet(os.path.join(PHASE2_DIR, "phase2_annotated.parquet"))
splits_path = os.path.join(PHASE3_DIR, "splits", "splits.json")
with open(splits_path) as f:
    splits = json.load(f)

test_idx = splits["test"]
df_test = df.iloc[test_idx].reset_index(drop=True)

# Subsample for benchmarking
if len(df_test) > BENCHMARK_NUM_SAMPLES:
    df_test = df_test.sample(n=BENCHMARK_NUM_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Benchmark test set: {len(df_test)} samples")
print(f"Languages: {BENCHMARK_LANGUAGES}")
print(f"Models: {list(BENCHMARK_MODELS.keys())}")

In [ ]:
# Cell 5: Editing model loader and image generation
from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler


def load_editing_model(model_name, model_id):
    """Load an InstructPix2Pix-style editing model."""
    print(f"Loading {model_name} ({model_id})...")
    try:
        pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            safety_checker=None,
        )
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
        pipe = pipe.to(device)
        pipe.set_progress_bar_config(disable=True)
        print(f"  Loaded successfully.")
        return pipe
    except Exception as e:
        print(f"  FAILED to load {model_name}: {e}")
        return None


def generate_edited_image(pipe, original_image, prompt, num_inference_steps=20, 
                          image_guidance_scale=1.5, guidance_scale=7.5):
    """Generate an edited image using the pipeline."""
    if isinstance(original_image, str):
        original_image = Image.open(original_image).convert("RGB")
    
    # Resize for pipeline (512x512)
    original_image = original_image.resize((512, 512), Image.LANCZOS)
    
    with torch.no_grad():
        result = pipe(
            prompt=prompt,
            image=original_image,
            num_inference_steps=num_inference_steps,
            image_guidance_scale=image_guidance_scale,
            guidance_scale=guidance_scale,
        )
    return result.images[0]


def get_prompt_for_language(row, lang):
    """Get the edit prompt in the specified language."""
    if lang == "en":
        return row["edit_prompt"]
    col = f"edit_prompt_{lang}"
    if col in row and pd.notna(row[col]):
        return row[col]
    return row["edit_prompt"]  # fallback to English


print("Editing model functions defined.")

In [ ]:
# Cell 6: Generate edited images for all models x languages
# Process ONE model at a time to fit in T4 memory

generation_log = {}

for model_name, model_id in BENCHMARK_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Generating with {model_name}")
    print(f"{'='*60}")
    
    pipe = load_editing_model(model_name, model_id)
    if pipe is None:
        print(f"  Skipping {model_name}")
        continue
    
    generation_log[model_name] = {}
    
    for lang in BENCHMARK_LANGUAGES:
        lang_dir = os.path.join(GEN_IMAGE_DIR, model_name, lang)
        os.makedirs(lang_dir, exist_ok=True)
        
        success_count = 0
        fail_count = 0
        
        for idx in tqdm(range(len(df_test)), desc=f"{model_name}/{lang}"):
            row = df_test.iloc[idx]
            sample_id = row.get("sample_id", f"{idx:06d}")
            output_path = os.path.join(lang_dir, f"{sample_id}.jpg")
            
            # Skip if already generated
            if os.path.exists(output_path):
                success_count += 1
                continue
            
            try:
                orig_path = os.path.join(PHASE1_DIR, row["original_image_path"])
                prompt = get_prompt_for_language(row, lang)
                
                edited_img = generate_edited_image(pipe, orig_path, prompt)
                edited_img.save(output_path, quality=95)
                success_count += 1
            except Exception as e:
                fail_count += 1
                if fail_count <= 3:
                    print(f"    Error on sample {sample_id}: {e}")
        
        generation_log[model_name][lang] = {
            "success": success_count,
            "failed": fail_count,
        }
        print(f"  {lang}: {success_count} success, {fail_count} failed")
    
    # Free memory before next model
    del pipe
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f"  Freed {model_name} from memory")

print("\nGeneration complete!")
print(json.dumps(generation_log, indent=2))

In [ ]:
# Cell 7: Quality metrics - CLIP-I, DINO, and classifier error rates
import open_clip

# Load CLIP for image similarity
print("Loading CLIP model...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(device).eval()

# Load DINO
print("Loading DINO model...")
dino_model = torch.hub.load("facebookresearch/dino:main", "dino_vits16")
dino_model = dino_model.to(device).eval()
dino_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


@torch.no_grad()
def compute_clip_i(img1, img2):
    """CLIP image-image similarity."""
    t1 = clip_preprocess(img1).unsqueeze(0).to(device)
    t2 = clip_preprocess(img2).unsqueeze(0).to(device)
    f1 = clip_model.encode_image(t1)
    f2 = clip_model.encode_image(t2)
    f1 = f1 / f1.norm(dim=-1, keepdim=True)
    f2 = f2 / f2.norm(dim=-1, keepdim=True)
    return (f1 * f2).sum().item()


@torch.no_grad()
def compute_dino_sim(img1, img2):
    """DINO image-image similarity."""
    t1 = dino_transform(img1).unsqueeze(0).to(device)
    t2 = dino_transform(img2).unsqueeze(0).to(device)
    f1 = dino_model(t1)
    f2 = dino_model(t2)
    f1 = f1 / f1.norm(dim=-1, keepdim=True)
    f2 = f2 / f2.norm(dim=-1, keepdim=True)
    return (f1 * f2).sum().item()


@torch.no_grad()
def classify_errors(orig_img, edit_img, prompt):
    """Run the trained error classifier on a single sample."""
    orig_t = eval_transform(orig_img).unsqueeze(0).to(device).half()
    edit_t = eval_transform(edit_img).unsqueeze(0).to(device).half()
    
    orig_feat = blip2_vision(pixel_values=orig_t).pooler_output.float()
    edit_feat = blip2_vision(pixel_values=edit_t).pooler_output.float()
    
    text_inputs = xlm_tokenizer(prompt, return_tensors="pt", padding=True,
                                 truncation=True, max_length=128).to(device)
    text_feat = xlm_model(**text_inputs).last_hidden_state[:, 0, :]
    
    logits = classifier(orig_feat, edit_feat, text_feat)
    probs = torch.sigmoid(logits).cpu().numpy()[0]
    preds = (probs > 0.5).astype(int)
    
    return preds, probs


print("Quality metric functions ready.")

In [ ]:
# Cell 8: Compute metrics for all model x language combinations
results = []

for model_name in BENCHMARK_MODELS:
    for lang in BENCHMARK_LANGUAGES:
        lang_dir = os.path.join(GEN_IMAGE_DIR, model_name, lang)
        if not os.path.exists(lang_dir):
            print(f"Skipping {model_name}/{lang} - no generated images")
            continue
        
        clip_scores, dino_scores = [], []
        error_rates, error_profiles = [], []
        
        for idx in tqdm(range(len(df_test)), desc=f"Eval {model_name}/{lang}"):
            row = df_test.iloc[idx]
            sample_id = row.get("sample_id", f"{idx:06d}")
            gen_path = os.path.join(lang_dir, f"{sample_id}.jpg")
            
            if not os.path.exists(gen_path):
                continue
            
            try:
                orig_path = os.path.join(PHASE1_DIR, row["original_image_path"])
                orig_img = Image.open(orig_path).convert("RGB")
                gen_img = Image.open(gen_path).convert("RGB")
                prompt = get_prompt_for_language(row, lang)
                
                # Image quality metrics
                clip_scores.append(compute_clip_i(orig_img, gen_img))
                dino_scores.append(compute_dino_sim(orig_img, gen_img))
                
                # Error classification
                preds, probs = classify_errors(orig_img, gen_img, prompt)
                has_error = int(preds.sum() > 0)
                error_rates.append(has_error)
                error_profiles.append(preds.tolist())
                
            except Exception as e:
                if len(error_rates) < 3:
                    print(f"  Error: {e}")
        
        if not clip_scores:
            continue
        
        # Aggregate metrics
        error_profile_avg = np.mean(error_profiles, axis=0) if error_profiles else np.zeros(NUM_ERROR_CLASSES)
        
        result = {
            "model": model_name,
            "language": lang,
            "clip_i": np.mean(clip_scores),
            "dino": np.mean(dino_scores),
            "error_rate": np.mean(error_rates),
            "n_samples": len(clip_scores),
            "error_profile": {ERROR_TAXONOMY[i]: float(v) for i, v in enumerate(error_profile_avg)},
        }
        results.append(result)
        
        print(f"  {model_name}/{lang}: CLIP-I={result['clip_i']:.4f}, "
              f"DINO={result['dino']:.4f}, error_rate={result['error_rate']:.4f}")

print(f"\nTotal results: {len(results)}")

In [ ]:
# Cell 9: Cross-lingual gap analysis and leaderboard
leaderboard_rows = []

for r in results:
    leaderboard_rows.append({
        "model": r["model"],
        "language": r["language"],
        "clip_i": round(r["clip_i"], 4),
        "dino": round(r["dino"], 4),
        "error_rate": round(r["error_rate"], 4),
        "n_samples": r["n_samples"],
    })

leaderboard = pd.DataFrame(leaderboard_rows)

# Compute cross-lingual gap
for model_name in BENCHMARK_MODELS:
    en_row = leaderboard[(leaderboard["model"] == model_name) & (leaderboard["language"] == "en")]
    if len(en_row) == 0:
        continue
    en_error_rate = en_row.iloc[0]["error_rate"]
    
    for lang in ["hi", "bn", "ne"]:
        lang_row = leaderboard[(leaderboard["model"] == model_name) & (leaderboard["language"] == lang)]
        if len(lang_row) == 0:
            continue
        lang_error_rate = lang_row.iloc[0]["error_rate"]
        
        if en_error_rate > 0:
            gap = (lang_error_rate - en_error_rate) / en_error_rate
        else:
            gap = 0.0
        
        idx = lang_row.index[0]
        leaderboard.loc[idx, "cross_lingual_gap"] = round(gap, 4)

# English has gap = 0
leaderboard.loc[leaderboard["language"] == "en", "cross_lingual_gap"] = 0.0

print("=" * 80)
print("BENCHMARK LEADERBOARD")
print("=" * 80)
print(leaderboard.to_string(index=False))
print()

# Summary by model
print("\nSummary by model (averaged across languages):")
summary = leaderboard.groupby("model").agg({
    "clip_i": "mean",
    "dino": "mean",
    "error_rate": "mean",
    "cross_lingual_gap": lambda x: x[x != 0].mean() if (x != 0).any() else 0,
}).round(4)
print(summary)

In [ ]:
# Cell 10: Save results
# Full benchmark results
benchmark_output = {
    "results": results,
    "generation_log": generation_log,
    "config": {
        "models": BENCHMARK_MODELS,
        "languages": BENCHMARK_LANGUAGES,
        "num_samples": BENCHMARK_NUM_SAMPLES,
    },
}

with open(os.path.join(OUTPUT_DIR, "benchmark_results.json"), "w") as f:
    json.dump(benchmark_output, f, indent=2, default=str)

# Leaderboard CSV
leaderboard.to_csv(os.path.join(OUTPUT_DIR, "leaderboard.csv"), index=False)

print(f"Results saved to {OUTPUT_DIR}/benchmark_results.json")
print(f"Leaderboard saved to {OUTPUT_DIR}/leaderboard.csv")

In [ ]:
# Cell 11: Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Leaderboard table as heatmap
models_list = leaderboard["model"].unique()
langs_list = leaderboard["language"].unique()
error_matrix = np.zeros((len(models_list), len(langs_list)))
for i, m in enumerate(models_list):
    for j, l in enumerate(langs_list):
        row = leaderboard[(leaderboard["model"] == m) & (leaderboard["language"] == l)]
        if len(row) > 0:
            error_matrix[i, j] = row.iloc[0]["error_rate"]

im = axes[0, 0].imshow(error_matrix, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=1)
axes[0, 0].set_xticks(range(len(langs_list)))
axes[0, 0].set_xticklabels(langs_list)
axes[0, 0].set_yticks(range(len(models_list)))
axes[0, 0].set_yticklabels(models_list)
for i in range(len(models_list)):
    for j in range(len(langs_list)):
        axes[0, 0].text(j, i, f"{error_matrix[i,j]:.3f}", ha="center", va="center", fontsize=9)
axes[0, 0].set_title("Error Rate by Model x Language")
plt.colorbar(im, ax=axes[0, 0])

# 2. Cross-lingual gap bar chart
gap_data = leaderboard[leaderboard["language"] != "en"].copy()
if len(gap_data) > 0:
    gap_pivot = gap_data.pivot(index="model", columns="language", values="cross_lingual_gap")
    gap_pivot.plot(kind="bar", ax=axes[0, 1], colormap="Set2")
    axes[0, 1].set_title("Cross-Lingual Gap (vs English)")
    axes[0, 1].set_ylabel("Relative Error Rate Increase")
    axes[0, 1].legend(title="Language")
    axes[0, 1].tick_params(axis="x", rotation=15)
    axes[0, 1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
else:
    axes[0, 1].text(0.5, 0.5, "No cross-lingual data", ha="center", va="center")
    axes[0, 1].set_title("Cross-Lingual Gap")

# 3. Per-model error profile radar chart
error_types = list(ERROR_TAXONOMY.values())
n_types = len(error_types)
angles = np.linspace(0, 2 * np.pi, n_types, endpoint=False).tolist()
angles += angles[:1]

ax_radar = fig.add_subplot(2, 2, 3, polar=True)
colors_radar = plt.cm.Set1(np.linspace(0, 1, len(models_list)))

for i, model_name in enumerate(models_list):
    model_results = [r for r in results if r["model"] == model_name and r["language"] == "en"]
    if not model_results:
        continue
    profile = model_results[0]["error_profile"]
    values = [profile.get(et, 0) for et in error_types]
    values += values[:1]
    ax_radar.plot(angles, values, "o-", linewidth=2, label=model_name, color=colors_radar[i])
    ax_radar.fill(angles, values, alpha=0.1, color=colors_radar[i])

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels([et[:10] for et in error_types], fontsize=7)
ax_radar.set_title("Error Profile (English)", pad=20)
ax_radar.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0), fontsize=8)
# Remove the default subplot at position (2,2,3) since we replaced it
axes[1, 0].set_visible(False)

# 4. CLIP-I and DINO comparison
x = np.arange(len(models_list))
width = 0.35
clip_means = [leaderboard[leaderboard["model"] == m]["clip_i"].mean() for m in models_list]
dino_means = [leaderboard[leaderboard["model"] == m]["dino"].mean() for m in models_list]

axes[1, 1].bar(x - width/2, clip_means, width, label="CLIP-I", color="steelblue")
axes[1, 1].bar(x + width/2, dino_means, width, label="DINO", color="coral")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(models_list, rotation=15)
axes[1, 1].set_ylabel("Similarity Score")
axes[1, 1].set_title("Image Quality Metrics (avg across languages)")
axes[1, 1].legend()
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "benchmark_visualization.png"), dpi=150, bbox_inches="tight")
plt.show()

print("=" * 60)
print("PHASE 4 COMPLETE")
print("=" * 60)
print(f"Output files:")
print(f"  - {OUTPUT_DIR}/benchmark_results.json")
print(f"  - {OUTPUT_DIR}/leaderboard.csv")
print(f"  - {OUTPUT_DIR}/benchmark_visualization.png")